# Sección 3: Verificación Numérica del Modelo Clásico
====================================================


Tesis: Teoría del Riesgo y Probabilidad de Ruina


Autor: Luis Alvarez
 
Base de datos: Insurance Claims (multi-ramo)


Filtro: Solo reclamaciones aprobadas (CLAIM_STATUS = 'A')


Unidad temporal: días


## Importar librerias

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.optimize import brentq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

## Definición de parámetros de las graficas

In [4]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'text.usetex': False,
})

# Para recreación
np.random.seed(123)


# Carga de Datos

In [5]:
df = pd.read_csv('insurance_data.csv')

df = df[df['CLAIM_STATUS'] == 'A'].copy()
df['REPORT_DT'] = pd.to_datetime(df['REPORT_DT'])
df['full_incident_timestamp'] = (
    df['REPORT_DT'] + pd.to_timedelta(df['INCIDENT_HOUR_OF_THE_DAY'], unit='h')
)
df = df.sort_values('full_incident_timestamp').reset_index(drop=True)
 
n_claims = len(df)
T_total_days = (
    df['full_incident_timestamp'].max() - df['full_incident_timestamp'].min()
).total_seconds() / 86400
 
print("=" * 65)
print("DESCRIPCIÓN DE LOS DATOS")
print("=" * 65)
print(f"  Registros aprobados:  {n_claims}")
print(f"  Horizonte:            {T_total_days:.1f} días ({T_total_days/30.44:.1f} meses)")
print(f"  Ramos:                {df['INSURANCE_TYPE'].nunique()}")

DESCRIPCIÓN DE LOS DATOS
  Registros aprobados:  9497
  Horizonte:            407.6 días (13.4 meses)
  Ramos:                6


# Estadísticas descriptivas de la severidad

In [6]:
claim_amounts = df['CLAIM_AMOUNT'].values
mu_Y = claim_amounts.mean()
std_Y = claim_amounts.std()
med_Y = np.median(claim_amounts)
skew_Y = stats.skew(claim_amounts)
kurt_Y = stats.kurtosis(claim_amounts)
EY2 = np.mean(claim_amounts**2)
 
print(f"\n=== Severidad (CLAIM_AMOUNT) ===")
print(f"  n           = {n_claims}")
print(f"  Media       = ${mu_Y:,.2f}")
print(f"  Desv. est.  = ${std_Y:,.2f}")
print(f"  Mediana     = ${med_Y:,.2f}")
print(f"  Mínimo      = ${claim_amounts.min():,.2f}")
print(f"  Máximo      = ${claim_amounts.max():,.2f}")
print(f"  Asimetría   = {skew_Y:.3f}")
print(f"  Curtosis    = {kurt_Y:.3f}")


=== Severidad (CLAIM_AMOUNT) ===
  n           = 9497
  Media       = $16,550.75
  Desv. est.  = $21,990.96
  Mediana     = $7,000.00
  Mínimo      = $100.00
  Máximo      = $100,000.00
  Asimetría   = 1.952
  Curtosis    = 3.359


# GRÁFICA 1: Histograma de severidad

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(claim_amounts, bins=60, density=True, alpha=0.7, color='#5C6BC0',
        edgecolor='white', linewidth=0.5)
# Ajuste exponencial superpuesto
delta_exp = 1 / mu_Y
x_exp = np.linspace(0, claim_amounts.max(), 500)
ax.plot(x_exp, delta_exp * np.exp(-delta_exp * x_exp), 'r-', linewidth=2,
        label=f'Exponencial ($\\delta = {delta_exp:.2e}$)')
ax.set_xlabel('Monto de reclamación (USD)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución empírica de la severidad de siniestros')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Graficas Tesis/fig_histograma_severidad.pdf')

# Reclamaciones por día y prueba de Poisso

In [7]:
claims_per_day = df.groupby(df['full_incident_timestamp'].dt.date).size()
cpd_values = claims_per_day.values
cpd_mean = cpd_values.mean()
cpd_var = cpd_values.var()
n_days = len(cpd_values)
 
print(f"\n=== Reclamaciones por día ===")
print(f"  Días observados = {n_days}")
print(f"  Media           = {cpd_mean:.2f}")
print(f"  Varianza        = {cpd_var:.2f}")
print(f"  Ratio var/media = {cpd_var/cpd_mean:.3f}")
 
# Chi-cuadrada para Poisson
from collections import Counter
freq_obs = Counter(cpd_values)
max_k = max(freq_obs.keys())
min_k = min(freq_obs.keys())
 
# Agrupar categorías con frecuencia esperada < 5
lambda_poi = cpd_mean
k_vals = np.arange(0, max_k + 5)
expected_probs = stats.poisson.pmf(k_vals, lambda_poi)
 
# Construir bins agrupados
bins_edges = []
bins_obs = []
bins_exp = []
current_obs = 0
current_exp = 0.0
 
for k in k_vals:
    obs_k = freq_obs.get(k, 0)
    exp_k = expected_probs[k] * n_days
    current_obs += obs_k
    current_exp += exp_k
    if current_exp >= 5:
        bins_obs.append(current_obs)
        bins_exp.append(current_exp)
        current_obs = 0
        current_exp = 0.0
 
# Agregar residuo
if current_obs > 0 or current_exp > 0:
    if len(bins_obs) > 0:
        bins_obs[-1] += current_obs
        bins_exp[-1] += current_exp
    else:
        bins_obs.append(current_obs)
        bins_exp.append(current_exp)
 
bins_obs = np.array(bins_obs)
bins_exp = np.array(bins_exp)
 
chi2_stat = np.sum((bins_obs - bins_exp)**2 / bins_exp)
dof = len(bins_obs) - 1 - 1  # -1 para lambda estimado
p_value_chi2 = 1 - stats.chi2.cdf(chi2_stat, dof)
 
print(f"\n  Prueba Chi-cuadrada (Poisson):")
print(f"  chi2 = {chi2_stat:.4f}")
print(f"  g.l. = {dof}")
print(f"  p-value = {p_value_chi2:.6f}")
print(f"  Resultado: {'NO rechazada' if p_value_chi2 > 0.05 else 'RECHAZADA'} (α=0.05)")


=== Reclamaciones por día ===
  Días observados = 409
  Media           = 23.22
  Varianza        = 35.65
  Ratio var/media = 1.535

  Prueba Chi-cuadrada (Poisson):
  chi2 = 63.0597
  g.l. = 20
  p-value = 0.000002
  Resultado: RECHAZADA (α=0.05)


# GRÁFICA 2: Reclamaciones por día vs Poisson teórica

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
k_range = np.arange(cpd_values.min() - 2, cpd_values.max() + 3)
hist_vals, bin_edges, _ = ax.hist(cpd_values, bins=np.arange(cpd_values.min()-0.5,
    cpd_values.max()+1.5, 1), density=False, alpha=0.7, color='#5C6BC0',
    edgecolor='white', linewidth=0.5, label='Datos observados')
 
# PMF teórica de Poisson
y_poisson = stats.poisson.pmf(k_range, lambda_poi) * n_days
ax.plot(k_range, y_poisson, 'ro-', markersize=4, linewidth=1.5,
        label=f'Poisson teórica ($\\lambda = {lambda_poi:.2f}$)')
ax.set_xlabel('Número de reclamaciones por día')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del número de reclamaciones diarias')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Graficas Tesis/fig_poisson_ajuste.pdf')

# Tiempos inter-arribo y prueba exponencial

In [8]:
inter_arrival = (
    df['full_incident_timestamp'].diff().dt.total_seconds().dropna() / 3600
)  # en horas
inter_arrival_pos = inter_arrival[inter_arrival > 0].values
ia_mean = inter_arrival_pos.mean()
ia_rate = 1 / ia_mean
 
print(f"\n=== Tiempos inter-arribo (horas, >0) ===")
print(f"  n           = {len(inter_arrival_pos)}")
print(f"  Media       = {ia_mean:.4f} horas")
print(f"  Desv. est.  = {inter_arrival_pos.std():.4f} horas")
 
# KS test para exponencial
ks_stat, ks_pval = stats.kstest(inter_arrival_pos, 'expon', args=(0, ia_mean))
print(f"\n  Prueba KS (Exponencial):")
print(f"  D = {ks_stat:.6f}")
print(f"  p-value = {ks_pval:.6e}")
print(f"  Resultado: {'NO rechazada' if ks_pval > 0.05 else 'RECHAZADA'} (α=0.05)")


=== Tiempos inter-arribo (horas, >0) ===
  n           = 6061
  Media       = 1.6141 horas
  Desv. est.  = 1.0828 horas

  Prueba KS (Exponencial):
  D = 0.461810
  p-value = 0.000000e+00
  Resultado: RECHAZADA (α=0.05)


# GRÁFICA 3: Histograma de tiempos inter-arribo vs exponencial

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(inter_arrival_pos, bins=50, density=True, alpha=0.7, color='#5C6BC0',
        edgecolor='white', linewidth=0.5, label='Datos observados')
x_ia = np.linspace(0, inter_arrival_pos.max(), 500)
ax.plot(x_ia, ia_rate * np.exp(-ia_rate * x_ia), 'r-', linewidth=2,
        label=f'Exponencial ($1/\\bar{{t}} = {ia_rate:.4f}$)')
ax.set_xlabel('Tiempo entre llegadas (horas)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de tiempos entre llegadas de siniestros')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Graficas Tesis/fig_interarribo_exponencial.pdf')
plt.close()

# Estimación de parámetros del modelo clásico

In [32]:
lambda_hat = n_claims / T_total_days  # siniestros/día
delta_hat = 1 / mu_Y                  # parámetro exponencial de severidad
theta_carga = 0.10                     # margen de seguridad
beta_hat = (1 + theta_carga) * lambda_hat * mu_Y  # USD/día
 
print(f"\n=== Parámetros del modelo clásico ===")
print(f"  lambda = {lambda_hat:.4f} sin/día")
print(f"  delta  = {delta_hat:.2e}")
print(f"  mu_Y   = ${mu_Y:,.2f}")
print(f"  theta  = {theta_carga}")
print(f"  beta   = ${beta_hat:,.2f} USD/día")
print(f"  E[S_1] = ${lambda_hat * mu_Y:,.2f} USD/día")
print(f"  Ganancia neta: beta - E[S_1] = ${beta_hat - lambda_hat * mu_Y:,.2f} > 0 ")


=== Parámetros del modelo clásico ===
  lambda = 23.2984 sin/día
  delta  = 6.04e-05
  mu_Y   = $16,550.75
  theta  = 0.1
  beta   = $424,166.21 USD/día
  E[S_1] = $385,605.64 USD/día
  Ganancia neta: beta - E[S_1] = $38,560.56 > 0 


# Coeficiente de ajuste R (exponencial)

In [33]:
R_CL = delta_hat - lambda_hat / beta_hat
print(f"\n  R_CL (analítico) = {R_CL:.6e}")
 
# Verificación por Brent
def lundberg_eq(R, lam, delta, beta_val):
    if R <= 0 or R >= delta:
        return np.inf
    return lam + beta_val * R - lam * delta / (delta - R)
 
R_brent = brentq(lundberg_eq, 1e-15, delta_hat - 1e-15,
                  args=(lambda_hat, delta_hat, beta_hat))
print(f"  R_CL (Brent)     = {R_brent:.6e}")
print(f"  Diferencia       = {abs(R_CL - R_brent):.2e}")
 
# Probabilidad de ruina en u=0
psi_0 = lambda_hat / (delta_hat * beta_hat)
print(f"  psi(0)           = {psi_0:.6f}")


  R_CL (analítico) = 5.492747e-06
  R_CL (Brent)     = 5.492747e-06
  Diferencia       = 1.62e-15
  psi(0)           = 0.909091


# Simulación Monte Carlo

In [34]:
T_horizon = 365 * 2   # 2 años en días
n_sim = 2_000
 
u_multiples = np.array([0, 1, 2, 5, 10, 15, 20])
u_values = u_multiples * mu_Y
 
print(f"\n=== Simulación Monte Carlo ===")
print(f"  Trayectorias: {n_sim}")
print(f"  Horizonte:    {T_horizon} días ({T_horizon/365:.0f} años)")
 
def simulate_CL_mc(u, beta_val, lam, delta, T, n_paths):
    """Simula CL path-by-path para ahorrar memoria."""
    ruined_count = 0
    for _ in range(n_paths):
        t_now, S_now = 0.0, 0.0
        is_ruined = False
        while t_now < T:
            t_now += np.random.exponential(1/lam)
            if t_now > T:
                break
            S_now += np.random.exponential(1/delta)
            if u + beta_val * t_now - S_now < 0:
                is_ruined = True
                break
        if is_ruined:
            ruined_count += 1
    return ruined_count / n_paths
 
results_mc = []
results_exact = []
results_lundberg = []
 
for u in u_values:
    psi_mc = simulate_CL_mc(u, beta_hat, lambda_hat, delta_hat, T_horizon, n_sim)
    psi_ex = (lambda_hat / (delta_hat * beta_hat)) * \
             np.exp(-(delta_hat - lambda_hat/beta_hat) * u)
    psi_lb = np.exp(-R_CL * u)
    results_mc.append(psi_mc)
    results_exact.append(psi_ex)
    results_lundberg.append(psi_lb)
    print(f"  u={u:>12,.0f} | MC={psi_mc:.5f} | Exacta={psi_ex:.5f} | Cota={psi_lb:.5f}")
 
results_mc = np.array(results_mc)
results_exact = np.array(results_exact)
results_lundberg = np.array(results_lundberg)


=== Simulación Monte Carlo ===
  Trayectorias: 2000
  Horizonte:    730 días (2 años)
  u=           0 | MC=0.89500 | Exacta=0.90909 | Cota=1.00000
  u=      16,551 | MC=0.82350 | Exacta=0.83009 | Cota=0.91310
  u=      33,102 | MC=0.75600 | Exacta=0.75796 | Cota=0.83375
  u=      82,754 | MC=0.57850 | Exacta=0.57703 | Cota=0.63474
  u=     165,508 | MC=0.35650 | Exacta=0.36626 | Cota=0.40289
  u=     248,261 | MC=0.24100 | Exacta=0.23248 | Cota=0.25573
  u=     331,015 | MC=0.15300 | Exacta=0.14756 | Cota=0.16232


# GRÁFICA 4: Trayectorias del superávit

In [39]:
fig, ax = plt.subplots(figsize=(8, 5))
u_traj = 200000
T_traj = 30  # 1 año
n_show = 5
 
np.random.seed(12345)
for k in range(n_show):
    t_events = [0.0]
    R_vals = [u_traj]
    t_now, S_now = 0.0, 0.0
    ruina_ocurrio = False
    
    while t_now < T_traj:
        dt_wait = np.random.exponential(1 / lambda_hat)
        t_now += dt_wait
        if t_now > T_traj:
            break
        claim = np.random.exponential(1 / delta_hat)
        S_now += claim
        # Punto antes del salto
        R_antes = u_traj + beta_hat * t_now - (S_now - claim)
        R_despues = u_traj + beta_hat * t_now - S_now
        
        t_events.append(t_now)
        R_vals.append(R_antes)
        # Punto después del salto
        t_events.append(t_now)
        R_vals.append(R_despues)
        
        if R_despues <= 0:
            ruina_ocurrio = True
            break # El proceso se detiene aquí
            
    # Configuración de estilo según el resultado
    estilo = ':' if ruina_ocurrio else '-'
    ancho = 1.2 if ruina_ocurrio else 0.8
    label_txt = f'Trayectoria {k+1} (ruina)' if ruina_ocurrio else f'Trayectoria {k+1}'
    
    ax.plot(t_events, R_vals, linestyle=estilo, linewidth=ancho, alpha=0.9, label=label_txt)
 
ax.axhline(0, color='red', linestyle='--', linewidth=1, label='Umbral de ruina')
ax.set_xlabel('Tiempo (días)')
ax.set_ylabel('Superávit $R_t$ (USD)')
ax.set_title(f'Trayectorias del proceso de superávit ($u_0 = \\${u_traj:,.0f}$)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Graficas Tesis/fig_trayectorias_CL.pdf')

C:\Users\Wisp8\AppData\Local\Temp\ipykernel_25244\3107963734.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(8, 5))


# GRÁFICA 5: Probabilidad de ruina (escala lineal)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
u_plot = np.linspace(0, 20 * mu_Y, 300)
psi_ex_plot = psi_0 * np.exp(-(delta_hat - lambda_hat/beta_hat) * u_plot)
cota_plot = np.exp(-R_CL * u_plot)
 
ax.plot(u_plot, psi_ex_plot, 'b-', linewidth=1.5, label='Fórmula exacta (Exp)')
ax.plot(u_plot, cota_plot, 'b--', linewidth=1, label='Cota de Lundberg')
ax.scatter(u_values, results_mc, marker='o', s=50, color='red',
           zorder=5, label='Monte Carlo')
ax.set_xlabel('Capital inicial $u$ (USD)')
ax.set_ylabel('$\\psi(u)$')
ax.set_title('Probabilidad de ruina — escala lineal')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.02, 1.05)
plt.tight_layout()
plt.savefig('Graficas Tesis/fig_psi_lineal.pdf')
plt.close()

# GRÁFICA 6: Probabilidad de ruina (escala logarítmica)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(u_plot, np.clip(psi_ex_plot, 1e-8, 1), 'b-', linewidth=1.5,
            label='Fórmula exacta (Exp)')
ax.semilogy(u_plot, np.clip(cota_plot, 1e-8, 1), 'b--', linewidth=1,
            label='Cota de Lundberg')
mask = results_mc > 0
ax.scatter(u_values[mask], results_mc[mask], marker='o', s=50, color='red',
           zorder=5, label='Monte Carlo')
ax.set_xlabel('Capital inicial $u$ (USD)')
ax.set_ylabel('$\\psi(u)$')
ax.set_title('Probabilidad de ruina — escala logarítmica')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
ax.set_ylim(1e-4, 2)
plt.tight_layout()
plt.savefig('Graficas Tesis/fig_psi_log.pdf')
plt.close()

# Tabla resumen para el LaTeX

In [ ]:
print("\n" + "=" * 65)
print("TABLA PARA LATEX")
print("=" * 65)
print(f"{'u (USD)':>15} | {'MC':>10} | {'Exacta':>10} | {'Cota':>10}")
print("-" * 55)
for i, u in enumerate(u_values):
    print(f"{u:>15,.0f} | {results_mc[i]:>10.5f} | {results_exact[i]:>10.5f} | {results_lundberg[i]:>10.5f}")

# Resumen de parámetros para LaTeX

In [ ]:
print(f"  n = {n_claims}")
print(f"  T = {T_total_days:.1f} días")
print(f"  lambda = {lambda_hat:.4f} sin/día")
print(f"  mu_Y = {mu_Y:,.2f}")
print(f"  delta = {delta_hat:.6e}")
print(f"  beta = {beta_hat:,.2f}")
print(f"  R = {R_CL:.6e}")
print(f"  psi(0) = {psi_0:.6f}")
print(f"  E[Y^2] = {EY2:,.0f}")
print(f"  Var[S_1] = {lambda_hat * EY2:,.0f}")